> **Chapter Map:** This notebook is the companion for **Chapter 5** (Selectivity Engineering).

**Prerequisites:** You should be comfortable with: (1) CSTR and PFR design equations from NB3, (2) Arrhenius temperature dependence from NB2.

# NB5: Selectivity Engineering

## Learning Objectives

After completing this notebook, you will be able to:

1. Simulate parallel and consecutive reaction networks in PFR and CSTR reactors
2. Calculate conversion, yield, and selectivity as functions of residence time
3. Identify the selectivity window for consecutive reactions and find the optimal residence time
4. Explain why PFR gives higher intermediate yield than CSTR for consecutive reactions
5. Use Arrhenius temperature dependence to optimize selectivity through temperature control
6. Show that N CSTRs in series approach PFR behavior as N increases

## Background

In real catalytic processes, the desired product is rarely the only one formed. **Selectivity engineering** addresses the question: *how do we maximize the yield of the product we want?*

Two fundamental reaction networks are studied in this notebook:

### Network 1: Parallel Reactions

$$\text{A} \xrightarrow{k_1} \text{B (desired)}, \qquad \text{A} \xrightarrow{k_2} \text{C (undesired)}$$

With first-order kinetics in A for both reactions:
$$r_1 = k_1 C_A, \quad r_2 = k_2 C_A$$

**Key result:** The selectivity $S_{B/C} = k_1/k_2$ is **independent of residence time**. Selectivity is controlled by **temperature** (via the Arrhenius equation).

### Network 2: Consecutive Reactions

$$\text{A} \xrightarrow{k_1} \text{B (desired)} \xrightarrow{k_2} \text{C (overproduct)}$$

**Key result:** The yield of B passes through a **maximum** at an optimal residence time $\tau^*$. This creates a **selectivity window** that depends on reactor type.

### Definitions

- **Conversion**: $X_A = 1 - C_A/C_{A0}$
- **Yield of B**: $Y_B = C_B/C_{A0}$
- **Selectivity to B (over C)**: $S_{B/C} = C_B/C_C$
- **Selectivity to B (from A)**: $S_{B/A} = C_B/(C_{A0} - C_A) = Y_B/X_A$

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

# Publication-quality plot defaults
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constant
R = 8.314  # Gas constant, J/(mol*K)

# Colorblind-safe color palette (Wong, 2011)
COLORS = {
    'blue': '#0072B2',
    'orange': '#E69F00',
    'green': '#009E73',
    'vermillion': '#D55E00',
    'skyblue': '#56B4E9',
    'purple': '#CC79A7'
}

print("Setup complete. Libraries imported successfully.")

---
## Part 1: Parallel Reactions in a PFR

For the parallel network A $\to$ B (desired) and A $\to$ C (undesired) with first-order kinetics, the PFR ODEs are:

$$\frac{dC_A}{d\tau} = -(k_1 + k_2)C_A$$

$$\frac{dC_B}{d\tau} = k_1 C_A$$

$$\frac{dC_C}{d\tau} = k_2 C_A$$

The analytical solutions are:

$$C_A(\tau) = C_{A0} \, e^{-(k_1+k_2)\tau}$$

$$C_B(\tau) = C_{A0} \frac{k_1}{k_1+k_2}\left(1 - e^{-(k_1+k_2)\tau}\right)$$

$$C_C(\tau) = C_{A0} \frac{k_2}{k_1+k_2}\left(1 - e^{-(k_1+k_2)\tau}\right)$$

The selectivity ratio $C_B/C_C = k_1/k_2$ is constant at all $\tau$.

In [ ]:
def parallel_pfr_ode(tau, y, k1, k2):
    """ODE system for parallel reactions A -> B (desired), A -> C (undesired) in a PFR.

    Parameters
    ----------
    tau : float
        Space time (s) -- independent variable
    y : array-like
        State vector [C_A, C_B, C_C] in mol/L
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for A -> C (s^-1)

    Returns
    -------
    list
        Derivatives [dCA/dtau, dCB/dtau, dCC/dtau]
    """
    C_A, C_B, C_C = y
    dCA = -(k1 + k2) * C_A
    dCB = k1 * C_A
    dCC = k2 * C_A
    return [dCA, dCB, dCC]


def parallel_pfr_analytical(tau, C_A0, k1, k2):
    """Analytical solution for parallel first-order reactions in a PFR.

    Parameters
    ----------
    tau : array-like
        Space time values (s)
    C_A0 : float
        Initial concentration of A (mol/L)
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for A -> C (s^-1)

    Returns
    -------
    C_A, C_B, C_C : ndarray
        Concentration profiles (mol/L)
    """
    tau = np.asarray(tau)
    k_total = k1 + k2
    C_A = C_A0 * np.exp(-k_total * tau)
    C_B = C_A0 * (k1 / k_total) * (1 - np.exp(-k_total * tau))
    C_C = C_A0 * (k2 / k_total) * (1 - np.exp(-k_total * tau))
    return C_A, C_B, C_C

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Try changing k1 and k2 to see how the ratio
# k1/k2 controls the selectivity split.
k1 = 0.4   # s^-1 (desired: A -> B)
k2 = 0.2   # s^-1 (undesired: A -> C), so k1/k2 = 2
C_A0 = 1.0 # mol/L
tau_max = 10.0  # s
# =============================================

tau_range = np.linspace(0, tau_max, 300)

# Analytical solution
C_A, C_B, C_C = parallel_pfr_analytical(tau_range, C_A0, k1, k2)

# Calculate performance metrics
X_A = 1 - C_A / C_A0           # Conversion
Y_B = C_B / C_A0               # Yield of B
Y_C = C_C / C_A0               # Yield of C
S_BA = np.where(X_A > 1e-10, Y_B / X_A, k1 / (k1 + k2))  # Selectivity B from A

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Concentration profiles
ax1.plot(tau_range, C_A, color=COLORS['blue'], linewidth=2.5,
         linestyle='-', label=r'$C_A$ (reactant)')
ax1.plot(tau_range, C_B, color=COLORS['green'], linewidth=2.5,
         linestyle='--', label=r'$C_B$ (desired)')
ax1.plot(tau_range, C_C, color=COLORS['vermillion'], linewidth=2.5,
         linestyle='-.', label=r'$C_C$ (undesired)')

# Mark asymptotic yields
Y_B_inf = k1 / (k1 + k2)
Y_C_inf = k2 / (k1 + k2)
ax1.axhline(y=Y_B_inf * C_A0, color=COLORS['green'],
            linestyle=':', linewidth=1, alpha=0.6)
ax1.axhline(y=Y_C_inf * C_A0, color=COLORS['vermillion'],
            linestyle=':', linewidth=1, alpha=0.6)
ax1.text(tau_max * 0.75, Y_B_inf * C_A0 + 0.03,
         f'$k_1/(k_1+k_2)$ = {Y_B_inf:.3f}', fontsize=10,
         color=COLORS['green'])
ax1.text(tau_max * 0.75, Y_C_inf * C_A0 + 0.03,
         f'$k_2/(k_1+k_2)$ = {Y_C_inf:.3f}', fontsize=10,
         color=COLORS['vermillion'])

ax1.set_xlabel(r'Space Time, $\tau$ (s)')
ax1.set_ylabel('Concentration (mol/L)')
ax1.set_title(f'Parallel Reactions in PFR\n$k_1$ = {k1} s$^{{-1}}$, '
              f'$k_2$ = {k2} s$^{{-1}}$')
ax1.set_xlim(0, tau_max)
ax1.set_ylim(0, 1.05)
ax1.legend(loc='center right', fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Yields and selectivity
ax2.plot(tau_range, Y_B, color=COLORS['green'], linewidth=2.5,
         linestyle='-', label=r'$Y_B = C_B/C_{A0}$')
ax2.plot(tau_range, Y_C, color=COLORS['vermillion'], linewidth=2.5,
         linestyle='--', label=r'$Y_C = C_C/C_{A0}$')
ax2.plot(tau_range, S_BA, color=COLORS['blue'], linewidth=2.5,
         linestyle='-.', label=r'$S_{B/A} = Y_B/X_A$')

ax2.set_xlabel(r'Space Time, $\tau$ (s)')
ax2.set_ylabel('Yield / Selectivity (dimensionless)')
ax2.set_title('Yields and Selectivity')
ax2.set_xlim(0, tau_max)
ax2.set_ylim(0, 1.05)
ax2.legend(loc='center right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Selectivity S_B/C = k1/k2 = {k1/k2:.2f} (constant at all tau)")
print(f"Selectivity S_B/A = k1/(k1+k2) = {k1/(k1+k2):.4f} (constant at all tau)")
print("\nKey insight: For parallel first-order reactions, selectivity is independent")
print("of residence time. It depends only on the ratio of rate constants.")

---
## Part 2: Consecutive Reactions -- The Selectivity Window

For the consecutive network A $\xrightarrow{k_1}$ B $\xrightarrow{k_2}$ C, the PFR ODEs are:

$$\frac{dC_A}{d\tau} = -k_1 C_A, \quad \frac{dC_B}{d\tau} = k_1 C_A - k_2 C_B, \quad \frac{dC_C}{d\tau} = k_2 C_B$$

The analytical PFR solution (for $k_1 \neq k_2$) is:

$$C_A(\tau) = C_{A0} e^{-k_1\tau}$$

$$C_B(\tau) = C_{A0} \frac{k_1}{k_2-k_1}\left(e^{-k_1\tau} - e^{-k_2\tau}\right)$$

$$C_C(\tau) = C_{A0} - C_A(\tau) - C_B(\tau)$$

The **optimal residence time** (maximum yield of B in PFR) is:

$$\tau^*_{\text{PFR}} = \frac{\ln(k_2/k_1)}{k_2 - k_1}$$

The CSTR steady-state solutions are:

$$C_A = \frac{C_{A0}}{1 + k_1\tau}, \qquad C_B = \frac{k_1\tau \cdot C_{A0}}{(1+k_1\tau)(1+k_2\tau)}$$

with **CSTR optimal residence time**:

$$\tau^*_{\text{CSTR}} = \frac{1}{\sqrt{k_1 k_2}}$$

In [ ]:
def consecutive_pfr_ode(tau, y, k1, k2):
    """ODE system for consecutive reactions A -> B -> C in a PFR.

    Parameters
    ----------
    tau : float
        Space time (s) -- independent variable
    y : array-like
        State vector [C_A, C_B, C_C] in mol/L
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    list
        Derivatives [dCA/dtau, dCB/dtau, dCC/dtau]
    """
    C_A, C_B, C_C = y
    dCA = -k1 * C_A
    dCB = k1 * C_A - k2 * C_B
    dCC = k2 * C_B
    return [dCA, dCB, dCC]


def consecutive_pfr_analytical(tau, C_A0, k1, k2):
    """Analytical solution for consecutive first-order reactions in a PFR.

    Parameters
    ----------
    tau : array-like
        Space time values (s)
    C_A0 : float
        Initial concentration of A (mol/L)
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    C_A, C_B, C_C : ndarray
        Concentration profiles (mol/L)

    Notes
    -----
    Valid for k1 != k2. For k1 = k2, use C_B = C_A0 * k1 * tau * exp(-k1*tau).
    """
    tau = np.asarray(tau)
    C_A = C_A0 * np.exp(-k1 * tau)

    if np.abs(k2 - k1) < 1e-12:  # k1 == k2 special case
        C_B = C_A0 * k1 * tau * np.exp(-k1 * tau)
    else:
        C_B = C_A0 * k1 / (k2 - k1) * (np.exp(-k1 * tau) - np.exp(-k2 * tau))

    C_C = C_A0 - C_A - C_B
    return C_A, C_B, C_C


def consecutive_cstr(tau, C_A0, k1, k2):
    """Steady-state solution for consecutive first-order reactions in a CSTR.

    Parameters
    ----------
    tau : array-like
        Space time values (s)
    C_A0 : float
        Inlet concentration of A (mol/L)
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    C_A, C_B, C_C : ndarray
        Exit concentrations (mol/L)
    """
    tau = np.asarray(tau)
    C_A = C_A0 / (1 + k1 * tau)
    C_B = k1 * tau * C_A0 / ((1 + k1 * tau) * (1 + k2 * tau))
    C_C = C_A0 - C_A - C_B
    return C_A, C_B, C_C


def tau_opt_pfr(k1, k2):
    """Optimal PFR residence time for maximum yield of B.

    Parameters
    ----------
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    float
        Optimal residence time (s)

    Notes
    -----
    tau* = ln(k2/k1) / (k2 - k1)  for k1 != k2
    tau* = 1/k1                     for k1 == k2
    """
    if np.abs(k2 - k1) < 1e-12:
        return 1.0 / k1
    return np.log(k2 / k1) / (k2 - k1)


def tau_opt_cstr(k1, k2):
    """Optimal CSTR residence time for maximum yield of B.

    Parameters
    ----------
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    float
        Optimal residence time (s)

    Notes
    -----
    tau* = 1 / sqrt(k1 * k2)
    """
    return 1.0 / np.sqrt(k1 * k2)

In [ ]:
# =============================================
# ADJUSTABLE PARAMETERS
# =============================================
# Consecutive reactions A -> B -> C.
# Try making k2 >> k1 or k1 >> k2 to see how
# the selectivity window changes.
k1_cons = 0.4   # s^-1 (A -> B)
k2_cons = 1.0   # s^-1 (B -> C)
C_A0 = 1.0      # mol/L
# =============================================

tau_range = np.linspace(0, 8, 400)

# PFR solution
C_A_pfr, C_B_pfr, C_C_pfr = consecutive_pfr_analytical(
    tau_range, C_A0, k1_cons, k2_cons)

# Optimal tau for PFR
tau_star_pfr = tau_opt_pfr(k1_cons, k2_cons)
_, C_B_max_pfr, _ = consecutive_pfr_analytical(
    tau_star_pfr, C_A0, k1_cons, k2_cons)
Y_B_max_pfr = C_B_max_pfr / C_A0

# Convert to yields
Y_B_pfr = C_B_pfr / C_A0
X_A_pfr = 1 - C_A_pfr / C_A0

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Concentration profiles
ax1.plot(tau_range, C_A_pfr, color=COLORS['blue'], linewidth=2.5,
         label=r'$C_A$ (reactant)')
ax1.plot(tau_range, C_B_pfr, color=COLORS['green'], linewidth=2.5,
         linestyle='--', label=r'$C_B$ (desired intermediate)')
ax1.plot(tau_range, C_C_pfr, color=COLORS['vermillion'], linewidth=2.5,
         linestyle='-.', label=r'$C_C$ (overproduct)')

# Mark the maximum of B
ax1.plot(tau_star_pfr, C_B_max_pfr, 'o', color=COLORS['green'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5,
         zorder=5)
ax1.annotate(f'$\\tau^*$ = {tau_star_pfr:.2f} s\n'
             f'$Y_{{B,max}}$ = {Y_B_max_pfr:.3f}',
             xy=(tau_star_pfr, C_B_max_pfr),
             xytext=(tau_star_pfr + 1.0, C_B_max_pfr + 0.15),
             fontsize=11,
             arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

# Shade the selectivity window
# Window: tau from 0.5*tau_star to 2*tau_star (approximate region of high Y_B)
tau_window_lo = tau_star_pfr * 0.3
tau_window_hi = tau_star_pfr * 3.0
ax1.axvspan(tau_window_lo, tau_window_hi, alpha=0.08,
            color=COLORS['green'])
ax1.text((tau_window_lo + tau_window_hi) / 2, 0.92,
         'Selectivity\nwindow', fontsize=10, ha='center',
         color=COLORS['green'], fontweight='bold')

# Annotations for too-short and too-long
ax1.annotate('Too short:\nlow $X_A$', xy=(0.3, 0.6), fontsize=10,
             ha='center', color='gray')
ax1.annotate('Too long:\nB overreacts', xy=(6.5, 0.6), fontsize=10,
             ha='center', color='gray')

ax1.set_xlabel(r'Space Time, $\tau$ (s)')
ax1.set_ylabel('Concentration (mol/L)')
ax1.set_title(f'Consecutive Reactions in PFR\n'
              f'$k_1$ = {k1_cons} s$^{{-1}}$, $k_2$ = {k2_cons} s$^{{-1}}$')
ax1.set_xlim(0, 8)
ax1.set_ylim(0, 1.05)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Yield of B vs conversion
ax2.plot(X_A_pfr, Y_B_pfr, color=COLORS['blue'], linewidth=2.5,
         label='PFR')

# Mark maximum yield point
X_A_at_star = 1 - np.exp(-k1_cons * tau_star_pfr)
ax2.plot(X_A_at_star, Y_B_max_pfr, 'o', color=COLORS['blue'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5)
ax2.annotate(f'($X_A$ = {X_A_at_star:.3f}, $Y_B$ = {Y_B_max_pfr:.3f})',
             xy=(X_A_at_star, Y_B_max_pfr),
             xytext=(X_A_at_star + 0.15, Y_B_max_pfr + 0.05),
             fontsize=11,
             arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

# 45-degree line (maximum possible yield if no overreaction)
X_line = np.linspace(0, 1, 100)
ax2.plot(X_line, X_line, ':', color='gray', linewidth=1,
         label=r'$Y_B = X_A$ (ideal, no overreaction)')

ax2.set_xlabel(r'Conversion, $X_A$')
ax2.set_ylabel(r'Yield of B, $Y_B$')
ax2.set_title('Yield--Conversion Tradeoff')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 0.5)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Optimal PFR residence time: tau* = {tau_star_pfr:.3f} s")
print(f"Maximum yield of B: Y_B,max = {Y_B_max_pfr:.3f} ({Y_B_max_pfr*100:.1f}%)")
print(f"Conversion at tau*: X_A = {X_A_at_star:.3f} ({X_A_at_star*100:.1f}%)")

---
## Part 3: PFR vs CSTR for Consecutive Reactions

The CSTR always gives a **lower maximum yield** of intermediate B than the PFR for consecutive reactions. The physical explanation is backmixing: in a CSTR, freshly-formed B is immediately mixed into the bulk, where it experiences the full average residence time and continues reacting to C. In a PFR, the plug flow profile protects early-formed B from over-reaction.

For the CSTR, the maximum yield is:

$$Y_{B,\text{max}}^{\text{CSTR}} = \frac{1}{\left(1 + \sqrt{k_2/k_1}\right)^2}$$

**Concept Check:** Predict: will the CSTR or PFR achieve higher maximum yield of intermediate B? Think about the effect of backmixing on consecutive reactions, then check the next cell.

In [ ]:
# CSTR solution
C_A_cstr, C_B_cstr, C_C_cstr = consecutive_cstr(
    tau_range, C_A0, k1_cons, k2_cons)

Y_B_cstr = C_B_cstr / C_A0
X_A_cstr = 1 - C_A_cstr / C_A0

# CSTR optimal
tau_star_cstr = tau_opt_cstr(k1_cons, k2_cons)
_, C_B_max_cstr, _ = consecutive_cstr(
    tau_star_cstr, C_A0, k1_cons, k2_cons)
Y_B_max_cstr = C_B_max_cstr / C_A0

# Also compute using the formula
Y_B_max_cstr_formula = 1.0 / (1 + np.sqrt(k2_cons / k1_cons))**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Yield vs tau
ax1.plot(tau_range, Y_B_pfr, color=COLORS['blue'], linewidth=2.5,
         label='PFR')
ax1.plot(tau_range, Y_B_cstr, color=COLORS['orange'], linewidth=2.5,
         linestyle='--', label='CSTR')

# Mark maxima
ax1.plot(tau_star_pfr, Y_B_max_pfr, 'o', color=COLORS['blue'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5)
ax1.plot(tau_star_cstr, Y_B_max_cstr, 's', color=COLORS['orange'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5)

ax1.annotate(f'PFR max: {Y_B_max_pfr:.3f}\nat $\\tau^*$ = {tau_star_pfr:.2f} s',
             xy=(tau_star_pfr, Y_B_max_pfr),
             xytext=(tau_star_pfr + 1.5, Y_B_max_pfr + 0.03),
             fontsize=10,
             arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=1.2))
ax1.annotate(f'CSTR max: {Y_B_max_cstr:.3f}\nat $\\tau^*$ = {tau_star_cstr:.2f} s',
             xy=(tau_star_cstr, Y_B_max_cstr),
             xytext=(tau_star_cstr + 2.0, Y_B_max_cstr - 0.04),
             fontsize=10,
             arrowprops=dict(arrowstyle='->', color=COLORS['orange'], lw=1.2))

ax1.set_xlabel(r'Space Time, $\tau$ (s)')
ax1.set_ylabel(r'Yield of B, $Y_B$')
ax1.set_title('Yield of Intermediate B vs Residence Time')
ax1.set_xlim(0, 8)
ax1.set_ylim(0, 0.30)
ax1.legend(loc='upper right', fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Yield vs Conversion (parametric plot)
ax2.plot(X_A_pfr, Y_B_pfr, color=COLORS['blue'], linewidth=2.5,
         label='PFR')
ax2.plot(X_A_cstr, Y_B_cstr, color=COLORS['orange'], linewidth=2.5,
         linestyle='--', label='CSTR')

# Mark maxima
X_A_pfr_star = 1 - np.exp(-k1_cons * tau_star_pfr)
X_A_cstr_star = 1 - 1 / (1 + k1_cons * tau_star_cstr)
ax2.plot(X_A_pfr_star, Y_B_max_pfr, 'o', color=COLORS['blue'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5)
ax2.plot(X_A_cstr_star, Y_B_max_cstr, 's', color=COLORS['orange'],
         markersize=12, markeredgecolor='black', markeredgewidth=1.5)

# Shade the PFR advantage region
ax2.fill_between(X_A_pfr, Y_B_pfr,
                 np.interp(X_A_pfr, X_A_cstr, Y_B_cstr),
                 alpha=0.15, color=COLORS['skyblue'],
                 label='PFR advantage')

# 45-degree line
ax2.plot(X_line, X_line, ':', color='gray', linewidth=1)

ax2.set_xlabel(r'Conversion, $X_A$')
ax2.set_ylabel(r'Yield of B, $Y_B$')
ax2.set_title('PFR vs CSTR: Yield--Conversion Diagram')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 0.30)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Comparison of Maximum B Yield")
print("=" * 50)
print(f"{'Reactor':>8} | {'tau* (s)':>10} | {'Y_B,max':>10} | {'X_A at tau*':>12}")
print("-" * 50)
print(f"{'PFR':>8} | {tau_star_pfr:>10.3f} | {Y_B_max_pfr:>10.3f} | {X_A_pfr_star:>12.3f}")
print(f"{'CSTR':>8} | {tau_star_cstr:>10.3f} | {Y_B_max_cstr:>10.3f} | {X_A_cstr_star:>12.3f}")
print(f"\nPFR advantage: {(Y_B_max_pfr/Y_B_max_cstr - 1)*100:.1f}% higher maximum yield")
print(f"CSTR formula check: Y_B,max = 1/(1+sqrt(k2/k1))^2 = {Y_B_max_cstr_formula:.3f}")

---
## Part 4: Temperature Effects on Selectivity

Temperature affects rate constants through the Arrhenius equation:

$$k_i(T) = A_i \exp\left(-\frac{E_{a,i}}{RT}\right)$$

The ratio $k_1/k_2$ therefore depends on temperature:

$$\frac{k_1}{k_2} = \frac{A_1}{A_2} \exp\left(-\frac{E_{a,1} - E_{a,2}}{RT}\right)$$

For **parallel reactions**, selectivity $S_{B/C} = k_1/k_2$, so:
- If $E_{a,1} > E_{a,2}$: **higher T improves selectivity** to B
- If $E_{a,1} < E_{a,2}$: **lower T improves selectivity** to B

For **consecutive reactions**, the situation is richer: temperature affects both the maximum yield $Y_{B,\text{max}}$ (through $k_1/k_2$) and the optimal residence time $\tau^*$.

In [ ]:
# Arrhenius parameters for parallel reactions (from Example 7.1)
A1_par = 1e10     # s^-1
Ea1_par = 80e3    # J/mol (desired reaction)
A2_par = 5e7      # s^-1
Ea2_par = 60e3    # J/mol (undesired reaction)

# Temperature sweep
T_range = np.linspace(350, 600, 200)  # K

k1_T = A1_par * np.exp(-Ea1_par / (R * T_range))
k2_T = A2_par * np.exp(-Ea2_par / (R * T_range))

S_BC = k1_T / k2_T  # Selectivity ratio

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Arrhenius plot of k1, k2
inv_T_1000 = 1000 / T_range
ax1.semilogy(inv_T_1000, k1_T, color=COLORS['green'], linewidth=2.5,
             label=f'$k_1$ ($E_{{a,1}}$ = {Ea1_par/1000:.0f} kJ/mol)')
ax1.semilogy(inv_T_1000, k2_T, color=COLORS['vermillion'], linewidth=2.5,
             linestyle='--',
             label=f'$k_2$ ($E_{{a,2}}$ = {Ea2_par/1000:.0f} kJ/mol)')

# Find crossover temperature where k1 = k2
# A1*exp(-Ea1/RT) = A2*exp(-Ea2/RT)
# T_cross = (Ea1-Ea2) / (R * ln(A1/A2))
T_cross = (Ea1_par - Ea2_par) / (R * np.log(A1_par / A2_par))
k_cross = A1_par * np.exp(-Ea1_par / (R * T_cross))
ax1.plot(1000 / T_cross, k_cross, 'ko', markersize=10, zorder=5)
ax1.annotate(f'Crossover: T = {T_cross:.0f} K',
             xy=(1000 / T_cross, k_cross),
             xytext=(1000 / T_cross - 0.15, k_cross * 10),
             fontsize=10,
             arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

ax1.set_xlabel('1000/$T$ (K$^{-1}$)')
ax1.set_ylabel(r'Rate Constant (s$^{-1}$)')
ax1.set_title('Arrhenius Plot of Competing Rates')
ax1.legend(loc='lower left', fontsize=10)
ax1.grid(True, alpha=0.3, which='both')

# Right: Selectivity vs Temperature
ax2.plot(T_range, S_BC, color=COLORS['blue'], linewidth=2.5)
ax2.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax2.axvline(x=T_cross, color='gray', linestyle=':', linewidth=1, alpha=0.7)

# Shade regions
ax2.fill_between(T_range, S_BC, 1.0,
                 where=(S_BC > 1), alpha=0.1, color=COLORS['green'])
ax2.fill_between(T_range, S_BC, 1.0,
                 where=(S_BC < 1), alpha=0.1, color=COLORS['vermillion'])

ax2.text(550, 2.5, 'B favored\n($S_{B/C} > 1$)', fontsize=11,
         ha='center', color=COLORS['green'], fontweight='bold')
ax2.text(380, 0.3, 'C favored\n($S_{B/C} < 1$)', fontsize=11,
         ha='center', color=COLORS['vermillion'], fontweight='bold')

ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel(r'Selectivity, $S_{B/C} = k_1/k_2$')
ax2.set_title(f'Selectivity vs Temperature\n'
              f'$E_{{a,1}}$ = {Ea1_par/1000:.0f} kJ/mol > '
              f'$E_{{a,2}}$ = {Ea2_par/1000:.0f} kJ/mol')
ax2.set_xlim(350, 600)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Since Ea,1 ({Ea1_par/1000:.0f} kJ/mol) > Ea,2 ({Ea2_par/1000:.0f} kJ/mol):")
print(f"  Higher temperature IMPROVES selectivity to B.")
print(f"  Crossover temperature: T = {T_cross:.0f} K")
print(f"  S_B/C at 400 K: {A1_par*np.exp(-Ea1_par/(R*400)) / (A2_par*np.exp(-Ea2_par/(R*400))):.2f}")
print(f"  S_B/C at 500 K: {A1_par*np.exp(-Ea1_par/(R*500)) / (A2_par*np.exp(-Ea2_par/(R*500))):.2f}")

In [ ]:
# Temperature effects on CONSECUTIVE reactions
# From Example 7.4 in lecture notes
A1_cons = 1e6      # s^-1
Ea1_cons = 50e3    # J/mol (A -> B)
A2_cons = 1e8      # s^-1
Ea2_cons = 70e3    # J/mol (B -> C)

T_sweep = np.linspace(350, 550, 100)  # K

# Calculate k1, k2, tau_opt, Y_B_max at each temperature
k1_sweep = A1_cons * np.exp(-Ea1_cons / (R * T_sweep))
k2_sweep = A2_cons * np.exp(-Ea2_cons / (R * T_sweep))
ratio_sweep = k1_sweep / k2_sweep

tau_opt_sweep = np.array([tau_opt_pfr(k1, k2)
                          for k1, k2 in zip(k1_sweep, k2_sweep)])

# Maximum yield from the analytical formula: Y_max = (k1/k2)^(k2/(k2-k1))
Y_B_max_sweep = np.array([
    (k1/k2)**(k2/(k2-k1)) if abs(k2-k1) > 1e-12 else 1/np.e
    for k1, k2 in zip(k1_sweep, k2_sweep)])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: k1/k2 ratio vs T
ax1.semilogy(T_sweep, ratio_sweep, color=COLORS['blue'], linewidth=2.5)
ax1.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_xlabel('Temperature (K)')
ax1.set_ylabel(r'$k_1/k_2$')
ax1.set_title(r'Rate Constant Ratio')
ax1.grid(True, alpha=0.3, which='both')

# Panel 2: Y_B,max vs T
ax2.plot(T_sweep, Y_B_max_sweep * 100, color=COLORS['green'], linewidth=2.5)
ax2.set_xlabel('Temperature (K)')
ax2.set_ylabel(r'$Y_{B,\max}$ (%)')
ax2.set_title('Maximum Yield of B')
ax2.grid(True, alpha=0.3)

# Panel 3: tau_opt vs T
ax3.semilogy(T_sweep, tau_opt_sweep, color=COLORS['orange'], linewidth=2.5)
ax3.set_xlabel('Temperature (K)')
ax3.set_ylabel(r'$\tau^*$ (s)')
ax3.set_title('Optimal Residence Time')
ax3.grid(True, alpha=0.3, which='both')

plt.suptitle(f'Consecutive Reactions: Temperature Sweep\n'
             f'$E_{{a,1}}$ = {Ea1_cons/1000:.0f} kJ/mol, '
             f'$E_{{a,2}}$ = {Ea2_cons/1000:.0f} kJ/mol',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Compare specific temperatures from lecture Example 7.4
for T_val in [400, 450]:
    k1_v = A1_cons * np.exp(-Ea1_cons / (R * T_val))
    k2_v = A2_cons * np.exp(-Ea2_cons / (R * T_val))
    tau_v = tau_opt_pfr(k1_v, k2_v)
    Y_v = (k1_v/k2_v)**(k2_v/(k2_v-k1_v))
    print(f"\nT = {T_val} K:")
    print(f"  k1 = {k1_v:.4f} s^-1, k2 = {k2_v:.4f} s^-1, k1/k2 = {k1_v/k2_v:.2f}")
    print(f"  tau* = {tau_v:.2f} s, Y_B,max = {Y_v:.3f} ({Y_v*100:.1f}%)")

print(f"\nSince Ea,2 > Ea,1: LOWER temperature gives HIGHER maximum yield.")
print(f"Tradeoff: lower T means longer tau* (larger reactor).")

---
## Part 5: N CSTRs in Series Approaching PFR

A cascade of $N$ equal-volume CSTRs in series approaches PFR behavior as $N \to \infty$. For consecutive reactions, this means the maximum yield of B improves progressively as more stages are added.

For each CSTR in the cascade with individual residence time $\tau_i = \tau_{\text{total}}/N$, we apply the CSTR mass balance sequentially:

$$C_{A,i} = \frac{C_{A,i-1}}{1 + k_1 \tau_i}, \qquad C_{B,i} = \frac{C_{B,i-1} + k_1 \tau_i \, C_{A,i}}{1 + k_2 \tau_i}$$

In [ ]:
def cstr_cascade_consecutive(tau_total, N, C_A0, k1, k2):
    """Solve N CSTRs in series for consecutive reactions A -> B -> C.

    Parameters
    ----------
    tau_total : float or array-like
        Total residence time across all N CSTRs (s)
    N : int
        Number of CSTRs in series
    C_A0 : float
        Inlet concentration of A (mol/L)
    k1 : float
        Rate constant for A -> B (s^-1)
    k2 : float
        Rate constant for B -> C (s^-1)

    Returns
    -------
    C_A_out, C_B_out, C_C_out : ndarray
        Exit concentrations from the last CSTR (mol/L)
    """
    tau_total = np.asarray(tau_total)
    C_A_out = np.zeros_like(tau_total)
    C_B_out = np.zeros_like(tau_total)

    for idx, tt in enumerate(tau_total):
        tau_i = tt / N  # Residence time per CSTR
        C_A = C_A0
        C_B = 0.0
        for _ in range(N):
            C_A_new = C_A / (1 + k1 * tau_i)
            C_B_new = (C_B + k1 * tau_i * C_A_new) / (1 + k2 * tau_i)
            C_A = C_A_new
            C_B = C_B_new
        C_A_out[idx] = C_A
        C_B_out[idx] = C_B

    C_C_out = C_A0 - C_A_out - C_B_out
    return C_A_out, C_B_out, C_C_out


# Compare different numbers of CSTRs in series
N_values = [1, 2, 3, 5]
tau_range_cascade = np.linspace(0.01, 8, 200)

fig, ax = plt.subplots(figsize=(10, 7))

# PFR reference (N -> infinity)
_, C_B_pfr_ref, _ = consecutive_pfr_analytical(
    tau_range_cascade, C_A0, k1_cons, k2_cons)
Y_B_pfr_ref = C_B_pfr_ref / C_A0
ax.plot(tau_range_cascade, Y_B_pfr_ref, color='black', linewidth=3,
        label=r'PFR ($N \to \infty$)')

# N CSTRs
cascade_colors = [COLORS['vermillion'], COLORS['orange'],
                  COLORS['green'], COLORS['blue']]
cascade_ls = [':', '-.', '--', '-']

for N, color, ls in zip(N_values, cascade_colors, cascade_ls):
    _, C_B_N, _ = cstr_cascade_consecutive(
        tau_range_cascade, N, C_A0, k1_cons, k2_cons)
    Y_B_N = C_B_N / C_A0
    Y_B_max_N = np.max(Y_B_N)
    ax.plot(tau_range_cascade, Y_B_N, color=color, linewidth=2,
            linestyle=ls,
            label=f'N = {N} CSTR{"s" if N > 1 else ""} '
                  f'($Y_{{B,max}}$ = {Y_B_max_N:.3f})')

ax.set_xlabel(r'Total Space Time, $\tau_{total}$ (s)')
ax.set_ylabel(r'Yield of B, $Y_B$')
ax.set_title('CSTRs in Series Approaching PFR Performance\n'
             f'$k_1$ = {k1_cons} s$^{{-1}}$, '
             f'$k_2$ = {k2_cons} s$^{{-1}}$')
ax.set_xlim(0, 8)
ax.set_ylim(0, 0.25)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("Maximum Yield of B: N CSTRs in Series")
print("=" * 40)
print(f"{'N':>5} | {'Y_B,max':>10}")
print("-" * 40)
for N in N_values:
    _, C_B_N, _ = cstr_cascade_consecutive(
        tau_range_cascade, N, C_A0, k1_cons, k2_cons)
    print(f"{N:>5} | {np.max(C_B_N/C_A0):>10.4f}")
print(f"{'PFR':>5} | {Y_B_max_pfr:>10.4f}")

### Maximum Yield vs Number of CSTRs

Let us quantify the convergence to PFR performance.

In [ ]:
N_range = np.arange(1, 21)
Y_max_vs_N = []

tau_fine = np.linspace(0.01, 10, 500)

for N in N_range:
    _, C_B_N, _ = cstr_cascade_consecutive(
        tau_fine, int(N), C_A0, k1_cons, k2_cons)
    Y_max_vs_N.append(np.max(C_B_N / C_A0))

Y_max_vs_N = np.array(Y_max_vs_N)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(N_range, Y_max_vs_N * 100, 'o-', color=COLORS['blue'],
        markersize=8, markeredgecolor='black', markeredgewidth=1,
        linewidth=2, label=r'$N$ CSTRs in series')
ax.axhline(y=Y_B_max_pfr * 100, color=COLORS['vermillion'],
           linestyle='--', linewidth=2,
           label=f'PFR limit ({Y_B_max_pfr*100:.1f}%)')
ax.axhline(y=Y_B_max_cstr * 100, color=COLORS['orange'],
           linestyle=':', linewidth=2,
           label=f'Single CSTR ({Y_B_max_cstr*100:.1f}%)')

ax.set_xlabel('Number of CSTRs in Series, $N$')
ax.set_ylabel(r'Maximum Yield $Y_{B,\max}$ (%)')
ax.set_title('Convergence to PFR Performance')
ax.set_xlim(0.5, 20.5)
ax.legend(loc='center right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"With 5 CSTRs: Y_B,max = {Y_max_vs_N[4]*100:.2f}% "
      f"(captures {(Y_max_vs_N[4]-Y_B_max_cstr)/(Y_B_max_pfr-Y_B_max_cstr)*100:.0f}% "
      f"of the PFR advantage over 1 CSTR)")

---
## Exercises

### Exercise 1: Temperature Control of Parallel Selectivity

For parallel reactions with $E_{a,1} > E_{a,2}$ (desired reaction has higher activation energy), should you operate at high or low temperature to maximize selectivity to B?

**Task:** Using the Arrhenius parameters from Part 4 ($E_{a,1} = 80$ kJ/mol, $E_{a,2} = 60$ kJ/mol), calculate $S_{B/C}$ at $T = 400$ K and $T = 500$ K. Verify your answer against the lecture notes.

In [ ]:
# YOUR CODE HERE
# Given: A1 = 1e10, Ea1 = 80e3, A2 = 5e7, Ea2 = 60e3
# Calculate k1(400K), k2(400K) and k1(500K), k2(500K)
# S_B/C = k1/k2 at each temperature
#
# Expected: S_B/C(400K) ~ 0.49, S_B/C(500K) ~ 1.63

pass  # Replace with your implementation

### Exercise 2: Optimal Residence Time for Maximum Yield

For consecutive reactions with $k_1 = 0.8$ s$^{-1}$ and $k_2 = 0.3$ s$^{-1}$ in a PFR:

**(a)** Calculate the optimal residence time $\tau^*$ using the analytical formula.

**(b)** Calculate the maximum yield $Y_{B,\text{max}}$ and the conversion $X_A$ at $\tau^*$.

**(c)** Plot the concentration profiles for A, B, and C versus $\tau$ and mark the selectivity window.

**Hint:** $\tau^* = \ln(k_2/k_1)/(k_2 - k_1)$ and $Y_{B,\max} = (k_1/k_2)^{k_2/(k_2-k_1)}$.

In [ ]:
# YOUR CODE HERE
# k1 = 0.8 s^-1, k2 = 0.3 s^-1, C_A0 = 1.0 mol/L
# (a) tau_star = ?
# (b) Y_B_max = ?, X_A at tau* = ?
# (c) Plot C_A, C_B, C_C vs tau

pass  # Replace with your implementation

### Exercise 3: Design for Selectivity and Conversion Targets

A pharmaceutical intermediate B is produced by the consecutive reaction A $\to$ B $\to$ C with $k_1 = 0.5$ s$^{-1}$ and $k_2 = 0.2$ s$^{-1}$ in a PFR.

The process specification requires:
- Selectivity to B: $S_{B/A} > 0.70$ (i.e., $Y_B/X_A > 0.70$)
- Conversion of A: $X_A > 0.50$

**Task:** Determine whether both specifications can be met simultaneously. If so, find the range of $\tau$ that satisfies both. If not, explain why and suggest which specification to relax.

**Hint:** Plot $S_{B/A}$ and $X_A$ vs $\tau$ on the same figure and find the intersection with the constraints.

In [ ]:
# YOUR CODE HERE
# k1 = 0.5, k2 = 0.2 s^-1, C_A0 = 1.0 mol/L
# Calculate S_B/A = Y_B / X_A and X_A vs tau
# Check if S_B/A > 0.70 AND X_A > 0.50 can be achieved simultaneously

pass  # Replace with your implementation

---
## Reflection Questions

1. For parallel first-order reactions, the selectivity $S_{B/C} = k_1/k_2$ is independent of residence time. Would this still be true if the two reactions had different reaction orders (e.g., $r_1 = k_1 C_A$ and $r_2 = k_2 C_A^2$)? What control lever would you gain?

2. In the CSTR, backmixing causes freshly-formed B to immediately experience the full residence time, reducing selectivity for consecutive reactions. In a real tubular reactor, axial dispersion creates some degree of backmixing. How would you expect the yield of B to change as axial dispersion increases from zero (ideal PFR) to infinity (ideal CSTR)?

3. For consecutive reactions with $E_{a,2} > E_{a,1}$ (overreaction has higher activation energy), lowering temperature improves $Y_{B,\text{max}}$ but increases $\tau^*$. In an industrial setting, what are the economic tradeoffs of this temperature-time relationship?

4. The cascade of N CSTRs converges to PFR performance as $N \to \infty$. In practice, what determines the optimal value of N for an industrial process? Consider both yield improvement and capital cost.

**Next notebook:** NB6 covers zeolites and carbon nanotubes (Chapter 6).